# Day 05 上午教师演示：电商用户行为数据分析

**课堂主线：** 数据验收 → 指标口径 → 用户画像 → 订单与优惠行为 → 多维分析 → 报表输出

> 本Notebook使用第4天生成的清洗后用户级数据。每行代表一名用户，不是一笔订单。

## 0. 使用说明与教学目标

完成本次演示后，学生应能够：

1. 使用 `groupby`、`agg`、`pivot_table` 完成单维与多维统计；
2. 同时报告样本量与比例；
3. 区分数据现象、业务解释和因果结论；
4. 输出可供下午项目和第6天可视化使用的统计报表。

**分析边界：** 当前数据没有订单金额和订单日期，因此不能计算GMV、客单价或时间趋势。

## 1. 环境、数据加载与验收

课堂提问：一行代表什么？`CustomerID`为什么不能求平均值？

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        # 优先查找 output/day04_project 路径
        if (candidate / "output" / "day04_project" / "ecommerce_customer_cleaned.csv").exists():
            return candidate
        # 备选：查找 data 文件夹
        if (candidate / "data" / "ecommerce_customer_cleaned.csv").exists():
            return candidate
    raise FileNotFoundError("未找到项目根目录，请确认第4天清洗结果已生成。")

ROOT = find_workspace_root()
# 根据实际存在的路径选择数据文件位置
if (ROOT / "output" / "day04_project" / "ecommerce_customer_cleaned.csv").exists():
    DATA_PATH = ROOT / "output" / "day04_project" / "ecommerce_customer_cleaned.csv"
else:
    DATA_PATH = ROOT / "data" / "ecommerce_customer_cleaned.csv"
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("项目根目录：", ROOT)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

In [ ]:
df = pd.read_csv(DATA_PATH)

core_cols = [
    "CustomerID", "Churn", "Tenure", "TenureGroup", "OrderCount",
    "CouponUsed", "CashbackAmount", "DaySinceLastOrder", "Complain",
    "PreferedOrderCat", "PreferredPaymentMode"
]

validation = pd.Series({
    "行数": len(df),
    "列数": df.shape[1],
    "CustomerID重复数": int(df["CustomerID"].duplicated().sum()),
    "核心字段缺失数": int(df[core_cols].isna().sum().sum()),
    "Churn取值": sorted(df["Churn"].unique().tolist()),
}, name="验收结果")

display(validation.to_frame())
display(df.head())

assert df.shape == (5630, 22), "数据形状与第4天交付物不一致"
assert df["CustomerID"].is_unique, "CustomerID存在重复"
assert df[core_cols].notna().all().all(), "核心字段仍有缺失"
assert set(df["Churn"].unique()) == {0, 1}, "Churn应只包含0和1"
print("数据验收通过：一行代表一名用户。")

### 教师讲解提示

- `CustomerID` 是标识符，只适合计数或去重计数。
- `Churn.mean()` 可以计算流失率，因为标签只取0和1。
- 分组流失率的分母是该分组的用户数，而不是全体用户数。

## 2. 指标字典：先定义再计算

In [ ]:
metric_dictionary = pd.DataFrame([
    ["用户数", "CustomerID", "nunique", "独立用户数量"],
    ["流失人数", "Churn", "sum", "Churn=1的用户数量"],
    ["流失率", "Churn", "mean", "当前分组中流失用户占比"],
    ["平均订单数", "OrderCount", "mean", "用户级平均订单次数"],
    ["平均优惠券数", "CouponUsed", "mean", "用户级平均优惠券使用次数"],
    ["平均返现", "CashbackAmount", "mean", "返现金额，不等于消费金额"],
    ["平均距上次下单天数", "DaySinceLastOrder", "mean", "越大通常表示近期活跃度越低"],
], columns=["指标名称", "字段", "聚合方式", "解释边界"])
display(metric_dictionary)

## 3. 总体用户画像

In [ ]:
overall_metrics = pd.DataFrame({
    "指标": ["用户数", "流失人数", "流失率", "平均订单数", "订单数中位数", "平均优惠券数", "平均返现", "平均App时长", "平均满意度", "平均距上次下单天数"],
    "数值": [
        df["CustomerID"].nunique(),
        df["Churn"].sum(),
        df["Churn"].mean(),
        df["OrderCount"].mean(),
        df["OrderCount"].median(),
        df["CouponUsed"].mean(),
        df["CashbackAmount"].mean(),
        df["HourSpendOnApp"].mean(),
        df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean(),
    ]
})
display(overall_metrics)
print(f"总体流失率：{df['Churn'].mean():.2%}")

In [ ]:
profile_fields = ["TenureGroup", "PreferedOrderCat", "PreferredPaymentMode", "PreferredLoginDevice", "CityTier"]
for field in profile_fields:
    table = df[field].value_counts(dropna=False).rename("用户数").to_frame()
    table["用户占比"] = table["用户数"] / len(df)
    print(f"\n--- {field} ---")
    display(table)

## 4. 用户生命周期、投诉与流失

In [ ]:
tenure_analysis = (
    df.groupby("TenureGroup", observed=True)
      .agg(
          用户数=("CustomerID", "nunique"),
          流失人数=("Churn", "sum"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
          平均返现=("CashbackAmount", "mean"),
          平均距上次下单天数=("DaySinceLastOrder", "mean"),
      )
      .reset_index()
)
display(tenure_analysis)

In [ ]:
complain_analysis = (
    df.groupby("Complain")
      .agg(
          用户数=("CustomerID", "nunique"),
          流失人数=("Churn", "sum"),
          流失率=("Churn", "mean"),
          平均满意度=("SatisfactionScore", "mean"),
          平均订单数=("OrderCount", "mean"),
      )
      .reset_index()
)
complain_analysis["投诉状态"] = complain_analysis["Complain"].map({0: "无投诉", 1: "有投诉"})
display(complain_analysis[["投诉状态", "用户数", "流失人数", "流失率", "平均满意度", "平均订单数"]])

### 结果解读练习

- 数据现象：投诉用户的流失率高于无投诉用户。
- 合理表达：当前样本中，投诉与流失存在明显关联。
- 不当表达：投诉必然导致用户流失。

注意观察满意度均值是否符合直觉。反直觉结果应被核查，而不是被隐藏。

## 5. 订单、品类与优惠行为分析

In [ ]:
category_analysis = (
    df.groupby("PreferedOrderCat")
      .agg(
          用户数=("CustomerID", "nunique"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
          平均优惠券数=("CouponUsed", "mean"),
          平均返现=("CashbackAmount", "mean"),
      )
      .reset_index()
      .sort_values(["流失率", "用户数"], ascending=[False, False])
)
category_analysis["用户占比"] = category_analysis["用户数"] / len(df)
display(category_analysis)

In [ ]:
payment_analysis = (
    df.groupby("PreferredPaymentMode")
      .agg(
          用户数=("CustomerID", "nunique"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
          平均优惠券数=("CouponUsed", "mean"),
          平均返现=("CashbackAmount", "mean"),
      )
      .reset_index()
      .sort_values("用户数", ascending=False)
)
display(payment_analysis)

In [ ]:
churn_behavior = (
    df.groupby("Churn")
      .agg(
          用户数=("CustomerID", "nunique"),
          平均订单数=("OrderCount", "mean"),
          平均优惠券数=("CouponUsed", "mean"),
          平均返现=("CashbackAmount", "mean"),
          平均App时长=("HourSpendOnApp", "mean"),
          平均满意度=("SatisfactionScore", "mean"),
          平均距上次下单天数=("DaySinceLastOrder", "mean"),
      )
      .reset_index()
)
churn_behavior["用户状态"] = churn_behavior["Churn"].map({0: "未流失", 1: "已流失"})
display(churn_behavior.drop(columns="Churn"))

### 课堂辨析

`CashbackAmount` 是返现金额，不是消费金额。当前数据只能讨论返现行为差异，不能计算销售额或客单价。

## 6. 多维分析：生命周期 × 投诉

In [ ]:
tenure_complain_analysis = (
    df.groupby(["TenureGroup", "Complain"], observed=True)
      .agg(
          用户数=("CustomerID", "nunique"),
          流失人数=("Churn", "sum"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
      )
      .reset_index()
)
tenure_complain_analysis["投诉状态"] = tenure_complain_analysis["Complain"].map({0: "无投诉", 1: "有投诉"})
tenure_complain_analysis["样本提示"] = np.where(tenure_complain_analysis["用户数"] < 30, "小样本", "可观察")
display(tenure_complain_analysis)

In [ ]:
count_pivot = pd.pivot_table(
    df, index="TenureGroup", columns="Complain", values="CustomerID",
    aggfunc="nunique", fill_value=0, observed=True
).rename(columns={0: "无投诉用户数", 1: "有投诉用户数"})

churn_pivot = pd.pivot_table(
    df, index="TenureGroup", columns="Complain", values="Churn",
    aggfunc="mean", observed=True
).rename(columns={0: "无投诉流失率", 1: "有投诉流失率"})

cross_pivot = count_pivot.join(churn_pivot).reset_index()
display(cross_pivot)

## 7. 报表输出与回读验证

In [ ]:
outputs = {
    "overall_metrics.csv": overall_metrics,
    "tenure_analysis.csv": tenure_analysis,
    "complain_analysis.csv": complain_analysis,
    "category_analysis.csv": category_analysis,
    "payment_analysis.csv": payment_analysis,
    "tenure_complain_analysis.csv": tenure_complain_analysis,
    "tenure_complain_pivot.csv": cross_pivot,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    check = pd.read_csv(path)
    print(f"已输出 {filename}: {check.shape}")

## 8. 课堂结论与出口条

请学生完成：

1. 写出总体流失率公式并说明分母；
2. 从生命周期或投诉分析中选择一条数据发现；
3. 把一条因果化表述改写为"关联＋待验证"的表达；
4. 说明当前数据为什么不能计算GMV或月度趋势。

**规范结论模板：** 在____用户中，____指标为____，与____相比____。当前样本显示____存在关联，可能与____有关；仍需结合____进一步验证。

---

### 参考答案

#### 1. 总体流失率公式与分母

**公式：** 流失率 = `Churn.sum() / CustomerID.nunique()` 或等价地 `df["Churn"].mean()`

**分母说明：** 分母是该数据集中的**全体用户数（5630人）**，即每一行代表一名独立用户。因为 Churn 字段仅取 0（未流失）或 1（已流失），对 Churn 字段求 `mean()` 等价于 `sum() / count()`，即流失用户占总用户的比例。分母不能是订单数或其他聚合指标，因为本数据集的观测单位是用户。

数值：948 / 5630 ≈ **16.84%**。

---

#### 2. 数据发现（生命周期分析）

选择 **TenureGroup（用户生命周期）** 维度的发现：

> 在新用户（0-6个月，1642人）中，流失率为 **25.88%**，平均订单数为 **2.68**，平均返现为 **164.87**；而在老用户（13-24个月，1467人）中，流失率仅为 **6.48%**，平均订单数为 **3.70**，平均返现为 **204.92**。

**现象总结：** 随着用户生命周期增长，流失率明显下降，同时平均订单数和返现金额均有所提升。

---

#### 3. 因果 → 关联改写

**❌ 因果化表述（不严谨）：**
> "使用E-wallet支付导致用户流失率更高。"

**✅ 关联＋待验证表述：**
> 在当前样本用户中，E-wallet 支付方式的流失率为 **23%**，高于 Debit Card（15%）和 Credit Card（14%），两者存在明显关联。但支付方式与流失之间的因果关系尚未验证——这种关联可能与选择 E-wallet 的用户群体本身的特征（如购买频次、品类偏好、App使用习惯等）有关；仍需结合用户分群和更细粒度的行为序列数据进一步验证。

---

#### 4. 为什么不能计算 GMV 或月度趋势

**不能计算 GMV 的原因：**
- 当前数据集中**没有订单金额（Order Amount/Revenue）字段**，也没有单价或商品价值相关字段。
- `CashbackAmount` 是**返现金额**，不是消费金额，二者在数值和业务含义上完全不同。
- 数据集是**用户级聚合数据**（每行一名用户），而非订单级明细数据，无法还原每笔交易的金额。

**不能计算月度趋势的原因：**
- 数据集中**没有订单日期（Order Date）或时间戳字段**。
- `Tenure` 是用户的累计生命周期月数，`DaySinceLastOrder` 是距上次下单天数——这两个字段只能反映时间间隔，不能还原具体的历史时间点。
- 没有时间维度就无法按月份进行分组聚合，因此无法绘制月度趋势图。

**总结：** 当前数据为横截面用户快照（cross-sectional user snapshot），适合做用户画像、分组对比和关联分析；但要做 GMV 分析或时间趋势分析，需要包含订单金额和订单日期的交易级明细数据。</cell id="cell-25">
